# Cross-lingual bridge: strict translation vs. paraphrase

This notebook extends the eng-vs-other-language similarity analysis to also
include paraphrase controls, per the L1/L2 x strict/paraphrase design:

- **L1**        = original English representation (the anchor, always from `data`/`results`)
- **strict**    = strict, structure-preserving translation into the target language (`data`/`results`)
- **l1_para**   = English paraphrase of the same excerpt (`data_p`/`results_p`, lang code `eng`)
- **l2_para**   = target-language paraphrase of the same excerpt (`data_p`/`results_p`, lang code = target language)

`data_p` / `results_p` reuse the exact same folder layout and language codes as
`data` / `results` — the only difference is that every language folder there
(including `eng`) holds a *paraphrase* instead of the original/strict-translation text.

In [ ]:
import os
import json
import pickle
from collections import defaultdict
from functools import reduce

import numpy as np
import pandas as pd
import torch
from datasets import Dataset

import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook"

pd.set_option('display.max_colwidth', None)

## IO helpers 

In [ ]:
def load_json(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_json(data, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

def load_pickle(filename):
    with open(filename, "rb") as pickle_handler:
        return pickle.load(pickle_handler)

def load_data(pii_type, filepath):
    """Load and lightly preprocess a PII dataset."""
    data = Dataset.load_from_disk(filepath)
    data = pd.DataFrame(data)
    data['context'] = data['context'].apply(str.strip)
    if len(data) > 4550 and pii_type == 'url':
        data = data.sample(n=4550, random_state=42).reset_index(drop=True)
    data = Dataset.from_pandas(data[['pii', 'context']])
    return data

def load(pii_type, dataset_path):
    return load_data(pii_type, dataset_path)

## Experiment configuration

In [ ]:
decoding_alg = "greedy"
torch.manual_seed(42)

model_types = {
    'Llama-3.2': ['1B', '3B'],
    'Qwen2.5': ['3B', "7B"]
}

contextToDo = "context-200"
pii_types = ["twitter"]#["phone"]#["email_cc"]#

# "eng" is the L1 anchor in the strict root, and the L1-paraphrase condition in the
# paraphrase root. fr/it/sp/de are the target languages (strict translation / L2 paraphrase).
languages = ["fr", "it", "sp", "de", "eng"]
target_languages = [l for l in languages if l != "eng"]

# Root 1: strict translations (this is the original notebook's "data" / "results")
ORIG_DATA_ROOT = "./data"
ORIG_RESULTS_ROOT = "./results"

# Root 2: paraphrases, SAME language codes and SAME folder layout as root 1.
# "eng" here = English paraphrase (L1 paraphrase); fr/it/sp/de here = target-language
# paraphrase (L2 paraphrase) -- NOT the strict translation.
PARA_DATA_ROOT = "./data_p"
PARA_RESULTS_ROOT = "./results_p"

BATCH_SIZE = 32

## 0. Path sanity check (both roots)

In [ ]:
def check_paths(data_root, results_root, label):
    print("=" * 80)
    print(f"Checking root: {label}  ({data_root} | {results_root})")
    print("=" * 80)
    ok = True
    for pii_type in pii_types:
        for lang in languages:
            dataset_path = f"{data_root}/Dataset-{pii_type}-{lang}"
            exists = os.path.exists(dataset_path)
            ok &= exists
            print(f"  [{'OK' if exists else 'MISSING'}] {dataset_path}")
    return ok

orig_ok = check_paths(ORIG_DATA_ROOT, ORIG_RESULTS_ROOT, "strict (orig)")
para_ok = check_paths(PARA_DATA_ROOT, PARA_RESULTS_ROOT, "paraphrase")

assert orig_ok, "Some strict-translation dataset paths are missing -- fix before continuing."
assert para_ok, "Some paraphrase dataset paths are missing -- fix before continuing."

## 1. Load activations

Generalizes the original notebook's loading logic to an arbitrary
`(data_root, results_root)` pair so the same code loads both the strict-translation
root and the paraphrase root. Only PII that were actually generated by the attack
(and whose saved activation file exists on disk) are kept.

In [ ]:
def load_root(data_root, results_root, label, pii_filter=None):
    """
    Parameters
    ----------
    pii_filter : dict or None
        If given, `pii_filter[pii_type][model]` is a set of pii strings. When set,
        loading is restricted to PII in this set, for EVERY language folder under
        this root -- rather than letting each language's own `generated.json`
        decide which PII to keep. This is how the paraphrase root is made to load
        "every activation corresponding to the strict version": the PII list comes
        from the strict-ENGLISH generated set, not from the paraphrase's own
        per-language generated files.
        Each language's own `generated.json` is still required to look up the
        on-disk index for a given PII (the activation filename encodes that
        index), so a PII is only loaded if it appears BOTH in `pii_filter` and in
        that language's own `generated` json.

    Returns
    -------
    activations : dict
        activations[pii_type][model][f"{lang}-{contextToDo}"][pii] -> hidden_states tensor [n_layers, hidden]
    generated_index : dict
        generated_index[pii_type][model][f"{lang}-{contextToDo}"] -> raw `generated` json (pii -> {"index": ...})
    full_pii_universe : dict
        full_pii_universe[pii_type][lang] -> set of every pii string in the SOURCE dataset
        (not just the ones successfully generated). Used for alignment checks below.
    errors : list
        Paths to activation files that were expected but not found on disk.
    """
    activations = {}
    generated_index = {}
    full_pii_universe = {}
    errors = []

    for pii_type in pii_types:
        print("*" * 80)
        print(f"[{label}] {pii_type}")
        print("*" * 80)

        activations[pii_type] = {}
        generated_index[pii_type] = {}
        full_pii_universe[pii_type] = {}

        for lang in languages:
            dataset_path = f"{data_root}/Dataset-{pii_type}-{lang}"
            data = load(pii_type, dataset_path)
            full_pii_universe[pii_type][lang] = set(data['pii'])
            print(f"  {lang}: {len(data)} rows, {len(full_pii_universe[pii_type][lang])} unique pii")

        for model_name, model_sizes in model_types.items():
            for model_size in model_sizes:
                model = f"{model_name}-{model_size}"
                activations[pii_type].setdefault(model, {})
                generated_index[pii_type].setdefault(model, {})

                # PII this (pii_type, model) is restricted to, or None = no restriction
                allowed_pii = None
                if pii_filter is not None:
                    allowed_pii = pii_filter[pii_type][model]

                for lang in languages:
                    lang_key = f"{lang}-{contextToDo}"
                    activations[pii_type][model][lang_key] = {}

                    folder_name = f"{results_root}/{pii_type}-{lang}"
                    context_results_dir = f"{folder_name}/results-{contextToDo}"
                    activation_results_dir = f"{folder_name}/activations-{contextToDo}/{model}-{decoding_alg}"

                    generated_file = f"{context_results_dir}/generated-{model}-{decoding_alg}.json"
                    generated = load_json(generated_file)
                    generated_index[pii_type][model][lang_key] = generated

                    candidate_pii = generated.keys() if allowed_pii is None else (generated.keys() & allowed_pii)

                    for pii in candidate_pii:
                        save_path = f"{activation_results_dir}/{pii}-{generated[pii]['index']}.pt"
                        if not os.path.exists(save_path):
                            errors.append(save_path)
                            continue
                        activation = torch.load(save_path, weights_only=False)
                        activations[pii_type][model][lang_key][pii] = activation['hidden_states']

                    msg = f"  {model} / {lang}: {len(activations[pii_type][model][lang_key])} activations loaded"
                    if allowed_pii is not None:
                        missing_from_generated = len(allowed_pii - generated.keys())
                        msg += f"  (filter had {len(allowed_pii)} pii; {missing_from_generated} not present in this language's generated.json)"
                    print(msg)
                    
                    

    return activations, generated_index, full_pii_universe, errors


def load_para_root(data_root, results_root, label, reference_generated):
    """
    Like `load_root`, but for the PARAPHRASE root only: the per-language
    `generated-*.json` file is NOT used to decide which pii to load. Instead,
    every language (including the English paraphrase) is loaded using the SAME
    pii -> index mapping taken from the strict-translation root's ENGLISH
    `generated` file (`reference_generated`), since paraphrase activations were
    saved under the same per-example index as the strict-translation dataset.

    Parameters
    ----------
    reference_generated : dict
        `generated_orig` as returned by `load_root` for the strict root --
        i.e. reference_generated[pii_type][model][f"eng-{contextToDo}"] gives
        the pii -> {"index": ...} mapping this function uses for every language.
    """
    activations = {}
    full_pii_universe = {}
    errors = []

    for pii_type in pii_types:
        print("*" * 80)
        print(f"[{label}] {pii_type}")
        print("*" * 80)

        activations[pii_type] = {}
        full_pii_universe[pii_type] = {}

        for lang in languages:
            dataset_path = f"{data_root}/Dataset-{pii_type}-{lang}"
            data = load(pii_type, dataset_path)
            full_pii_universe[pii_type][lang] = set(data['pii'])
            print(f"  {lang}: {len(data)} rows, {len(full_pii_universe[pii_type][lang])} unique pii")

        for model_name, model_sizes in model_types.items():
            for model_size in model_sizes:
                model = f"{model_name}-{model_size}"
                activations[pii_type].setdefault(model, {})

                # Fixed selection for every language below: the pii that were
                # generated by the STRICT ENGLISH attack for this model.
                reference = reference_generated[pii_type][model][f"eng-{contextToDo}"]

                for lang in languages:
                    lang_key = f"{lang}-{contextToDo}"
                    activations[pii_type][model][lang_key] = {}

                    folder_name = f"{results_root}/{pii_type}-{lang}"
                    activation_results_dir = f"{folder_name}/activations-{contextToDo}/{model}-{decoding_alg}"

                    activation_save_paths = set(os.listdir(f"{activation_results_dir}"))
                    for pii, info in reference.items():
                        
                        save_paths = list(filter(lambda x:  x.startswith(f'{pii}-'), activation_save_paths))
                        if not save_paths:
                            errors.append(f"{activation_results_dir}/{pii}")
                            continue
                        save_path = f"{activation_results_dir}/{save_paths[0]}"
                        activation = torch.load(save_path, weights_only=False)
                        activations[pii_type][model][lang_key][pii] = activation['hidden_states']

                    print(f"  {model} / {lang}: {len(activations[pii_type][model][lang_key])} activations loaded (ref=English strict generated)")

    return activations, full_pii_universe, errors





activations_orig, generated_orig, pii_universe_orig, errors_orig = load_root(
    ORIG_DATA_ROOT, ORIG_RESULTS_ROOT, "strict"
)
activations_para, pii_universe_para, errors_para = load_para_root(
    PARA_DATA_ROOT, PARA_RESULTS_ROOT, "paraphrase", generated_orig
)

if errors_orig:
    print(f"WARNING: {len(errors_orig)} expected strict-translation activation files were missing.")
if errors_para:
    print(f"WARNING: {len(errors_para)} expected paraphrase activation files were missing.")

## 2. Alignment checks

Since alignment can only go through the `pii` string (there's no shared id column),
two things need checking before the similarity numbers can be trusted:

1. **Duplicate PII within a dataset** -- if the same `pii` string is attached to more
   than one excerpt, `pii` alone is not a safe join key for that pii type/language.
2. **Source overlap between the strict root and the paraphrase root** -- the paraphrase
   dataset for a given language should be a reworded version of (close to) the SAME
   underlying excerpts as the strict dataset. Low overlap means the paraphrase set was
   drawn from a different sample and per-example comparison is not meaningful.

In [ ]:
def report_duplicate_pii(data_root, label):
    print(f"--- Duplicate-pii check: {label} ({data_root}) ---")
    for pii_type in pii_types:
        for lang in languages:
            dataset_path = f"{data_root}/Dataset-{pii_type}-{lang}"
            data = pd.DataFrame(load(pii_type, dataset_path))
            n_total = len(data)
            n_unique = data['pii'].nunique()
            n_dup = n_total - n_unique
            flag = "  <-- pii is NOT a unique key here" if n_dup > 0 else ""
            print(f"  {pii_type}/{lang}: {n_total} rows, {n_unique} unique pii, {n_dup} duplicated{flag}")

report_duplicate_pii(ORIG_DATA_ROOT, "strict")
report_duplicate_pii(PARA_DATA_ROOT, "paraphrase")

In [ ]:
def report_source_overlap(pii_universe_orig, pii_universe_para):
    print("--- Source-overlap check (strict vs paraphrase pii universes) ---")
    for pii_type in pii_types:
        for lang in languages:
            s_orig = pii_universe_orig[pii_type][lang]
            s_para = pii_universe_para[pii_type][lang]
            inter = s_orig & s_para
            frac_of_orig = len(inter) / max(len(s_orig), 1)
            frac_of_para = len(inter) / max(len(s_para), 1)
            warn = "  <-- LOW OVERLAP, check this is the same underlying sample" if frac_of_orig < 0.9 else ""
            print(
                f"  {pii_type}/{lang}: orig={len(s_orig)}, para={len(s_para)}, "
                f"shared={len(inter)} ({frac_of_orig:.1%} of orig, {frac_of_para:.1%} of para){warn}"
            )

report_source_overlap(pii_universe_orig, pii_universe_para)

## 3. Build the 4-way shared PII sets, per model / target language

For each target language, ALL three curves (strict, l1_para, l2_para) must be computed
on the exact same set of examples, so they're a fair comparison. That requires PII that
were successfully generated in all four conditions:

- strict-translation attack in English      (`activations_orig[...]['eng-...']`)
- strict-translation attack in the target lang (`activations_orig[...][f'{lang}-...']`)
- paraphrase attack in English               (`activations_para[...]['eng-...']`)
- paraphrase attack in the target lang        (`activations_para[...][f'{lang}-...']`)

In [ ]:
shared_pii = {}  # shared_pii[pii_type][model][lang] -> sorted list of pii strings

for pii_type in pii_types:
    shared_pii[pii_type] = {}
    for model_name, model_sizes in model_types.items():
        for model_size in model_sizes:
            model = f"{model_name}-{model_size}"
            shared_pii[pii_type][model] = {}

            s_l1 = set(activations_orig[pii_type][model][f"eng-{contextToDo}"].keys())
            s_l1_para = set(activations_para[pii_type][model][f"eng-{contextToDo}"].keys())

            print(f"[{pii_type} | {model}] L1={len(s_l1)}  L1_para={len(s_l1_para)}")

            for lang in target_languages + ['eng']:
                s_l2 = set(activations_orig[pii_type][model][f"{lang}-{contextToDo}"].keys())
                s_l2_para = set(activations_para[pii_type][model][f"{lang}-{contextToDo}"].keys())

                shared = s_l1 & s_l2 & s_l1_para & s_l2_para
                shared_pii[pii_type][model][lang] = sorted(shared)

                print(
                    f"    {lang}: L2={len(s_l2)}  L2_para={len(s_l2_para)}  "
                    f"-> 4-way shared={len(shared)}"
                )

## 4. Stack row-aligned activation tensors per condition

In [ ]:
condition_tensors = {}  # condition_tensors[pii_type][model][lang] = {"pii":..., "L1":..., "strict":..., "l1_para":..., "l2_para":...}

for pii_type in pii_types:
    condition_tensors[pii_type] = {}
    for model_name, model_sizes in model_types.items():
        for model_size in model_sizes:
            model = f"{model_name}-{model_size}"
            condition_tensors[pii_type][model] = {}

            for lang in target_languages:
                piis = shared_pii[pii_type][model][lang]
                if not piis:
                    print(f"WARNING: no 4-way-shared pii for {pii_type}/{model}/{lang} -- skipping")
                    continue
                
                X_L1 = torch.stack([activations_orig[pii_type][model][f"eng-{contextToDo}"][p] for p in piis])
                X_strict = torch.stack([activations_orig[pii_type][model][f"{lang}-{contextToDo}"][p] for p in piis])
                X_l1_para = torch.stack([activations_para[pii_type][model][f"eng-{contextToDo}"][p] for p in piis])
                X_l2_para = torch.stack([activations_para[pii_type][model][f"{lang}-{contextToDo}"][p] for p in piis])

                condition_tensors[pii_type][model][lang] = {
                    "pii": piis,
                    "L1": X_L1,
                    "strict": X_strict,
                    "l1_para": X_l1_para,
                    "l2_para": X_l2_para,
                }

            piis = shared_pii[pii_type][model]['eng']
            X_L1 = torch.stack([activations_orig[pii_type][model][f"eng-{contextToDo}"][p] for p in piis])
            X_l1_para = torch.stack([activations_para[pii_type][model][f"eng-{contextToDo}"][p] for p in piis])
            
            condition_tensors[pii_type][model]['eng'] = {
                "pii": piis,
                "L1": X_L1,
                "strict": X_strict,
                "l1_para": X_l1_para,
                "l2_para": X_l2_para,
            }

## 5. Per-layer paired cosine similarity

Each condition tensor is already row-aligned to L1, so this is a row-wise cosine between
`X_L1[i]` and `X_cond[i]`, averaged over `i`, computed independently per layer. 

In [ ]:
def paired_cosine_per_layer(X_L1, X_cond):
    """
    X_L1, X_cond: [n_examples, n_layers, hidden]
    Returns: np.array of shape [n_layers] with mean cosine similarity at each layer.
    """
    a = X_L1.float()
    b = X_cond.float()
    num = (a * b).sum(dim=-1)
    denom = a.norm(dim=-1) * b.norm(dim=-1)
    cos = num / denom.clamp_min(1e-8)   # [n_examples, n_layers]
    return cos.mean(dim=0).numpy()      # [n_layers]


similarity_scores = {}  # similarity_scores[model][lang] -> DataFrame(layer, strict, l1_para, l2_para)

for pii_type in pii_types:
    for model in condition_tensors[pii_type]:
        similarity_scores.setdefault(model, {})
        for lang, tensors in condition_tensors[pii_type][model].items():
            if lang == 'eng':
                sim_l1_para = paired_cosine_per_layer(tensors["L1"], tensors["l1_para"])
                df = pd.DataFrame({
                    "layer": np.arange(len(sim_l1_para)),
                    "l1_para": sim_l1_para,
                    
                })
                df.attrs["n_examples"] = len(tensors["pii"])
                similarity_scores[model][lang] = df
            else:
                sim_strict = paired_cosine_per_layer(tensors["L1"], tensors["strict"])
                sim_l2_para = paired_cosine_per_layer(tensors["L1"], tensors["l2_para"])
    
                df = pd.DataFrame({
                    "layer": np.arange(len(sim_strict)),
                    "strict": sim_strict,
                    "l2_para": sim_l2_para,
                })
                df.attrs["n_examples"] = len(tensors["pii"])
                similarity_scores[model][lang] = df
        

## 6. Save per-layer scores to CSV

In [ ]:
os.makedirs("results_p/similarity_scores", exist_ok=True)
for model, per_lang in similarity_scores.items():
    for lang, df in per_lang.items():
        out_path = f"results_p/similarity_scores/{str(pii_types)}-{model}-{lang}.csv"
        df.to_csv(out_path, index=False)

## 7. Combined plot: strict / L1-paraphrase / L2-paraphrase together, per model

One figure per model. Color = target language, line style = condition
(solid = strict translation, dot = English paraphrase, dash = target-language paraphrase).

In [ ]:
CONDITION_STYLE = {
    "strict": "solid",
    "l1_para": "dot",
    "l2_para": "dash",
}
CONDITION_LABEL = {
    "strict": "Strict translation",
    "l1_para": "English paraphrase (L1)",
    "l2_para": "Target-language paraphrase (L2)",
}

lang_palette = px.colors.qualitative.Plotly

for model, per_lang in similarity_scores.items():
    fig = go.Figure()
    for i, lang in enumerate(target_languages + ['eng']):
        if lang not in per_lang:
            continue
        df = per_lang[lang]
        color = lang_palette[i % len(lang_palette)]
        n = df.attrs.get("n_examples", "?")
        for cond in ["strict", "l2_para"] if lang != 'eng' else ["l1_para"]:
            fig.add_trace(go.Scatter(
                x=df["layer"],
                y=df[cond],
                mode="lines",
                name=f"{lang} ({n}) - {CONDITION_LABEL[cond]}",
                line=dict(color=color, width=2.5, dash=CONDITION_STYLE[cond]),
                legendgroup=lang,
            ))

    fig.update_layout(
        title=f"{model}: cosine similarity to L1 (English) representation, per layer",
        xaxis_title="layer",
        yaxis_title="mean paired cosine similarity",
        width=1050,
        height=650,
        template="plotly_white",
        hovermode="x unified",
    )
    fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


CONDITION_STYLE = {
    "strict": "solid",
    "l1_para": "dash",
    "l2_para": "dot"
}

CONDITION_LABEL = {
    "strict": "Strict translation",
    "l1_para": "English paraphrase",
    "l2_para": "Paraphrase",
}

# Colour-blind friendly Okabe-Ito palette
LANG_COLORS = {
    "fr": "#0072B2",   # blue
    "it": "#E69F00",   # orange
    "sp": "#009E73",   # green
    "de": "#CC79A7",   # purple
    "eng": "#D55E00",  # vermillion
}

LANG_LABEL = {
    "fr": "French",
    "it": "Italian",
    "sp": "Spanish",
    "de": "German",
    "eng": "English",
}


to_test_models = {
    'Qwen2.5',
    'Llama'
}

models = [x for x in similarity_scores.keys() if x.split('-')[0] in to_test_models]

fig = make_subplots(
    rows=1,
    cols=len(models),
    shared_xaxes=True,
    shared_yaxes=False,
    vertical_spacing=0.08,
    subplot_titles=[m.replace('-', ' ') for m in models],
)

for row, model in enumerate(models, start=1):
    per_lang = similarity_scores[model]

    for lang in target_languages + ["eng"]:

        if lang not in per_lang:
            continue

        df = per_lang[lang]
        n = df.attrs.get("n_examples", "?")

        # English only has the L1 paraphrase condition
        conditions = (
            ["l1_para"]
            if lang == "eng"
            else ["strict", "l2_para"]
        )

        for cond in conditions:

            fig.add_trace(
                go.Scatter(
                    x=df["layer"],
                    y=df[cond],
                    mode="lines",
                    name=f"{LANG_LABEL.get(lang, lang)} - "
                         f"{CONDITION_LABEL[cond]}",
                    legendgroup=f"{lang}_{cond}",
                    showlegend=(row == 1),
                    line=dict(
                        color=LANG_COLORS.get(lang, "#444444"),
                        width=2.5 if cond != 'l1_para' else 4,
                        dash=CONDITION_STYLE[cond],
                    ),
                    hovertemplate=(
                        f"<b>{LANG_LABEL.get(lang, lang)}</b>"
                        f"<br>{CONDITION_LABEL[cond]}"
                        f"<br>Layer: %{{x}}"
                        f"<br>Cosine: %{{y:.3f}}"
                        "<extra></extra>"
                    ),
                ),
                row=1,
                col=row
            )

    # Individual y-axis label
    fig.update_yaxes(
        #title_text="Mean paired cosine similarity",
        row=1,
        col=row,
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)",
        zeroline=False,
    )

    fig.update_xaxes(
        title_text="Layer",
        row=1,
        col=row,
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)",
        zeroline=False,
    )

fig.update_layout(
    width=300 * len(models),
    height=500,
    template="plotly_white",

    # Put the legend below the figure
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.3,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255,255,255,0.9)",
        borderwidth=0,
    ),

    hovermode="x unified",

    margin=dict(
        l=90,
        r=40,
        t=90,
        b=150,
    ),
)
fig.write_html(f"bridge_{str(pii_types)}.html")
fig.show()

In [ ]:
exit()